#  추천 시스템 - Text2SQL 기반 RAG 시스템

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Langsmith tracing 여부를 확인 (true: langsmith 추척 활성화, false: langsmith 추척 비활성화)
import os
print(os.getenv('LANGSMITH_TRACING'))

`(3) SQLite DB`

In [1]:
from langchain_community.utilities import SQLDatabase
import ast

db = SQLDatabase.from_uri("sqlite:///etf_database.db")
print(db.dialect)
print(db.get_usable_table_names())
etfs = db.run("SELECT * FROM ETFs LIMIT 5;")

for etf in ast.literal_eval(etfs):
    print(etf)

sqlite
['ETFs', 'ETFsInfo']
('466400', '1Q 25-08 회사채(A+이상)액티브', '2023/09/19', '채권-회사채-단기', '하나자산운용', 4.52, 'KIS 2025-08만기형 크레딧 A+이상 지수(총수익)', 0.11, 111916276404.0, 0.03, '매우낮음', '실물(액티브)', 0.1, '배당소득세(보유기간과세)')
('491610', '1Q CD금리액티브(합성)', '2024/09/24', '기타', '하나자산운용', 0.0, 'KIS 하나 CD금리 총수익지수', 0.05, 316206006696.0, 0.02, '매우낮음', '합성(액티브)', 0.02, '배당소득세(보유기간과세)')
('451060', '1Q K200액티브', '2023/01/31', '주식-시장대표', '하나자산운용', -3.66, '코스피 200', 0.77, 99754348820.0, -0.01, '높음', '실물(액티브)', 0.18, '배당소득세(보유기간과세)')
('463290', '1Q 단기금융채액티브', '2023/08/03', '채권-혼합-단기', '하나자산운용', 4.01, 'MK 머니마켓 지수(총수익)', 0.05, 252717462257.0, 0.0, '매우낮음', '실물(액티브)', 0.08, '배당소득세(보유기간과세)')
('479080', '1Q 머니마켓액티브', '2024/04/02', '채권-혼합-단기', '하나자산운용', 0.0, 'KIS-하나 MMF 지수(총수익)', 0.06, 308255065986.0, -0.01, '매우낮음', '실물(액티브)', 0.05, '배당소득세(보유기간과세)')


---

## **SQL QA** 

- **SQL 기반 Q&A 시스템**은 구조화된 데이터를 LLM으로 처리하는 특수 사례
    - **자연어 처리**를 통한 SQL 쿼리 자동 변환 구현
    - 데이터베이스에서 **쿼리 실행 및 결과 추출** 
    - 추출된 데이터를 활용한 **자연어 답변 생성**

- **참조**: https://python.langchain.com/docs/tutorials/sql_qa/

`(1) State 상태 정의`

- **LangGraph 상태 관리**를 위한 TypedDict 구조 정의

- 입력 **질문**, **쿼리**, **결과**, **답변**의 4가지 핵심 상태 추적

- 상태 데이터의 단계별 전달로 실행 흐름 제어

In [ ]:
from typing import TypedDict

# 상태 정보를 저장하는 State 클래스
class State(TypedDict):
    question: str  # 입력 질문
    query: str     # 생성된 쿼리
    result: str    # 쿼리 결과
    answer: str    # 생성된 답변

`(2) 프롬프트 템플릿`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from typing_extensions import TypedDict, Annotated

# 프롬프트 템플릿 생성
query_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
    Given an input question, create a syntactically correct {dialect} query to run to help find the answer. 
    Unless the user specifies in his question a specific number of examples they wish to obtain, 
    always limit your query to at most {top_k} results.
      
    You can order the results by a relevant column to return the most interesting examples in the database.
    Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.
    Pay attention to use only the column names that you can see in the schema description. 
    Be careful to not query for columns that do not exist.
      
    Also, pay attention to which column is in which table.
    When joining tables, you MUST use explicit table prefixes for all columns (e.g. table_name.column_name) to avoid ambiguous column errors.\n",
    Only use the following tables:
    {table_info}
    """),
    ("user", """
    Question:
    {input}
    """)
])

# 프롬프트 템플릿 출력
for m in query_prompt_template.messages:
   m.pretty_print()

In [ ]:
# required 입력 필드 확인 (dialect, input, table_info, top_k)
query_prompt_template.input_schema.model_json_schema()

`(3) SQL 쿼리 생성`

In [ ]:
from langchain_openai import ChatOpenAI

# llm 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from typing import Annotated, TypedDict

class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]


def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(),
            "input": state["question"],
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

In [ ]:
# 쿼리 실행
response = write_query({"question": "총보수가 0.1% 이하인 ETF는 무엇인가요?"}) 

response

`(4) SQL 쿼리 실행`

In [ ]:
from langchain_community.tools import QuerySQLDatabaseTool

def execute_query(state: State):
    """Execute SQL query."""
    try:
        execute_query_tool = QuerySQLDatabaseTool(db=db)
        return {"result": execute_query_tool.invoke(state["query"])}
    except Exception as e:
        return {"result": f"쿼리 실행 중 오류 발생: {str(e)}"}

In [ ]:
execute_query({"query": response["query"]})

`(5) RAG 답변 생성`

In [ ]:
def generate_answer(state: State):
    """Answer question using retrieved information as context."""
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question.\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}

In [ ]:
question = "총보수가 0.1% 이하인 ETF는 무엇인가요?"

query = write_query({"question": question})["query"]  #type: ignore
result = execute_query({"query": query})["result"]    #type: ignore

response = generate_answer({
    "question": question,
    "query": query,
    "result": result,
})   #type: ignore

print(response["answer"])

`(6) LangGraph 통합`

- langgraph 설치 (pip 또는 uv)

In [ ]:
from langgraph.graph import START, StateGraph 

graph_builder = StateGraph(State)

graph_builder.add_node("write_query", write_query)
graph_builder.add_node("execute_query", execute_query)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "write_query")
graph_builder.add_edge("write_query", "execute_query")
graph_builder.add_edge("execute_query", "generate_answer")  
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
for step in graph.stream(
    {"question": "총보수가 0.1% 이하인 ETF는 모두 몇 개인가요?"}, stream_mode="updates"
):
    print(step)

## **DB 테이블 관리** 

- **테이블 선택** 및 **다중 테이블 처리** 기능으로 유연한 데이터 관리
- 자연어로 표현된 질문을 **SQL로 자동 변환** 및 실행
- 테이블 관리와 SQL 실행을 통합한 지능형 데이터베이스 시스템 구현

`(1) 특정 테이블만 선택`

- **Sakila Sample Database**
- 출처: https://www.kaggle.com/datasets/atanaskanev/sqlite-sakila-sample-database/data

In [ ]:
from langchain_community.utilities import SQLDatabase

# SQLite 데이터베이스 연결
db = SQLDatabase.from_uri("sqlite:///sqlite-sakila.db", sample_rows_in_table_info=3)

# 데이터베이스 정보 확인
print(db.dialect)
print(db.get_usable_table_names())

sqlite
[]


: 

In [ ]:
from langchain_community.utilities import SQLDatabase

# 일부 테이블 제외
simple_db = SQLDatabase.from_uri(
    "sqlite:///sqlite-sakila.db",
    ignore_tables=['actor', 'address', 'category', 'city', 'country', 'customer', 'inventory', 'language', 'payment', 'rental', 'staff', 'store'],
    )
print(simple_db.dialect)

# 사용 가능한 테이블 목록
print(simple_db.get_usable_table_names())

In [ ]:
import ast

films = db.run("SELECT * FROM film LIMIT 5;")

for etf in ast.literal_eval(films):
    print(etf)

`(2) 복잡한 DB 스키마 처리`

- 테이블들을 Film, Customer, Location 3개의 카테고리로 그룹화
- Pydantic 모델을 사용하여 테이블 구조 정의

In [ ]:
# 데이터베이스 정보 확인
print(db.dialect)
print(db.get_usable_table_names())

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from operator import itemgetter
from typing import List

# 테이블 카테고리 정의를 위한 Pydantic 모델
class Table(BaseModel):
    """SQL 데이터베이스의 테이블."""
    name: str = Field(description="SQL 데이터베이스의 테이블 이름")
    category: str = Field(description="테이블이 속하는 카테고리")
    purpose: str = Field(description="테이블의 주요 목적 및 역할")

class TableList(BaseModel):
    """SQL 데이터베이스의 테이블 목록."""
    tables: List[Table] = Field(description="관련된 테이블들의 목록")

# LLM 모델 초기화 
llm = ChatOpenAI(model='gpt-4.1-mini')

# 실제 SQLite Sakila 테이블에 맞춘 시스템 프롬프트
system = """당신은 Sakila DVD 대여점 데이터베이스의 전문가입니다.
사용자 질문과 관련된 SQL 테이블들을 식별하고 적절한 카테고리로 분류하세요.

## 실제 SQLite Sakila 데이터베이스 테이블 구조:
테이블 목록: ['actor', 'address', 'category', 'city', 'country', 'customer', 'film', 'film_actor', 'film_category', 'film_text', 'inventory', 'language', 'payment', 'rental', 'staff', 'store']

### Film (영화 및 콘텐츠)
- **film**: 영화 기본 정보 
- **film_category**: 영화-카테고리 관계 테이블 
- **film_actor**: 영화-배우 관계 테이블 
- **category**: 영화 장르/카테고리 
- **language**: 영화 언어 정보 
- **actor**: 배우 정보 

### Customer (고객 및 직원)
- **customer**: 고객 정보 
- **staff**: 직원 정보 

### Transaction (거래 및 대여)
- **rental**: 대여 기록 
- **payment**: 결제 정보 

### Inventory (재고 관리)
- **inventory**: 영화 재고 정보 

### Location (위치 및 매장)
- **store**: 매장 정보 
- **address**: 주소 정보 
- **city**: 도시 정보 
- **country**: 국가 정보

## 테이블 간 주요 관계:
- **customer** → **rental** → **payment**: 고객이 영화를 대여하고 결제
- **film** → **inventory** → **rental**: 영화 재고를 통한 대여 관리
- **film** ↔ **actor** (film_actor): 영화와 배우 다대다 관계
- **film** ↔ **category** (film_category): 영화와 장르 다대다 관계
- **store** → **inventory**: 매장별 재고 관리
- **address** → **city** → **country**: 계층적 주소 구조
- **staff** → **store**: 직원과 근무 매장 관계

## 분석 시 고려사항:
- 대여 분석: rental, customer, inventory, film 테이블 조합 필요
- 매출 분석: payment, rental, customer 테이블 조합 필요  
- 영화 검색: film, film_text, actor, category, language 테이블 활용
- 지역별 분석: store, address, city, country 테이블 조합 필요

사용자 질문을 분석하여 관련된 테이블들을 식별하고, 각 테이블의 카테고리와 목적을 명시하세요."""

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "질문: {input}\n\n위 질문에 답하기 위해 필요한 테이블들을 식별하고 분류해주세요."),
])

# LLM 구조화 출력 설정
structure_llm = llm.with_structured_output(TableList)

# 카테고리 체인 생성
category_chain = prompt | structure_llm 

In [ ]:
# Customer 카테고리 체인 실행 
question = "어떤 고객이 가장 많은 영화를 대여했나요?"
table_list = category_chain.invoke({"input": question})

for table in table_list.tables:
    print(f"📋 테이블: {table.name}")
    print(f"   카테고리: {table.category}")
    print(f"   목적: {table.purpose}")
    print()

In [ ]:
# Film 카테고리 체인 실행
question = "가장 인기 있는 영화 장르는 무엇인가요?"
table_list = category_chain.invoke({"input": question})

for table in table_list.tables:
    print(f"📋 테이블: {table.name}")
    print(f"   카테고리: {table.category}")
    print(f"   목적: {table.purpose}")
    print()

In [ ]:
# Location 카테고리 체인 실행
question = "각 매장의 총 매출은 얼마인가요?"
table_list = category_chain.invoke({"input": question})

for table in table_list.tables:
    print(f"📋 테이블: {table.name}")
    print(f"   카테고리: {table.category}")
    print(f"   목적: {table.purpose}")
    print()


- **모든 테이블 사용**

In [ ]:
from typing import Annotated, TypedDict

class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]

# SQL 쿼리 체인 생성
partial_prompt_template = query_prompt_template.format(
    dialect=db.dialect,
    top_k=10,
    table_info="{table_names_to_use}",
    input="{question}"
)

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template(partial_prompt_template)

structured_llm = llm.with_structured_output(QueryOutput)
query_chain = prompt | structured_llm

# SQL 쿼리 생성 - 모든 테이블 사용
response = query_chain.invoke({
    "question": "가장 많은 작품을 대여한 고객은 누구인가요?",
    "table_names_to_use": db.get_table_info()})

print(response.get("query"))


In [ ]:
# SQL 쿼리 실행
db.run(response.get("query"))

- **일부 테이블 사용**

In [ ]:
# 테이블 이름 체인 생성 (테이블 이름 추출)
table_chain = (
    {"input": itemgetter("question")} 
    | category_chain 
    | RunnableLambda(lambda x: [t.name for t in x.tables])
)

# 테이블 이름 체인 실행
table_chain.invoke({"question": "가장 많은 작품을 대여한 고객은 누구인가요?"})

In [ ]:
# 최종 체인 연결 (테이블 이름 추출 -> SQL 쿼리 생성) -> 추출된 일부 테이블 정보를 사용하여 쿼리 생성
full_chain = RunnablePassthrough.assign(
    table_names_to_use=table_chain | RunnableLambda(lambda x: db.get_table_info(x))
    ) | query_chain

response = full_chain.invoke({"question": "가장 많은 작품을 대여한 고객은 누구인가요?"})
print(response.get("query"))

In [ ]:
# SQL 쿼리 실행
db.run(response.get("query")) 

- **Agent로 구현**

In [ ]:
from langchain_core.tools import tool
from typing import List, Annotated, TypedDict
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

### 1. 테이블 선택 도구 생성

@tool  
def select_relevant_tables(question: str) -> str:
    """질문과 관련된 테이블의 스키마 정보를 반환한다."""
    # 기존 table_chain 결과를 직접 사용
    try:
        selected_tables = table_chain.invoke({"question": question})
        table_info = db.get_table_info(selected_tables)
        return f"관련 테이블: {', '.join(selected_tables)}\n\n{table_info}"
    except Exception as e:
        return f"테이블 선택 중 오류 발생: {str(e)}"


### 2. Agent 도구 준비

# 기본 SQL 도구킷 생성
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

# 커스텀 테이블 선택 도구 추가
custom_tools = [select_relevant_tables]  

# 모든 도구 결합
all_tools = sql_tools + custom_tools

# 시스템 메시지 정의 (테이블 선택 기능 포함)
system_message = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {db.dialect} query to run, 
then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, 
always limit your query to at most 5 results.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.

IMPORTANT WORKFLOW:
1. FIRST, use the 'select_relevant_tables' tool to identify relevant tables for the question
2. THEN, if needed, use 'sql_db_list_tables' to see all available tables
3. THEN, use 'sql_db_schema' to get detailed schema for specific tables
4. THEN, use 'sql_db_query_checker' to validate your query before execution
5. FINALLY, use 'sql_db_query' to execute the validated query

You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

Never query for all the columns from a specific table, only ask for the relevant columns given the question.
"""


### 3. StateGraph로 Agent 구조화

# 상태(State) 정의
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 모델 노드 실행 로직
def call_model(state: AgentState):
    messages = state["messages"]
    # 시스템 메시지를 첫 번째 시스템 롤로 설정
    msg_to_send = [{"role": "system", "content": system_message}] + messages
    
    # 모델에 도구 바인딩 & 실행
    llm_with_tools = llm.bind_tools(all_tools)
    response = llm_with_tools.invoke(msg_to_send)
    
    return {"messages": [response]}

# 다음 단계(노드)를 결정하는 라우팅 함수
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    # 도구 호출(tool_calls)이 있으면 도구 노드로 이동, 없으면 종료
    if last_message.tool_calls:
        return "tools"
    return END

# 그래프 생성
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(all_tools))

# 엣지 추가 (흐름 정의)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

# 그래프 컴파일
app = workflow.compile()


### 4. 실행
question = "가장 많은 작품을 대여한 고객은 누구인가요?"

for step in app.stream(
    {"messages": [HumanMessage(content=question)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()


In [ ]:
# Film 카테고리 체인 실행
question = "가장 인기 있는 영화 장르는 무엇인가요?"

for step in app.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
# Location 카테고리 체인 실행
question = "각 매장의 총 매출은 얼마인가요?"

for step in app.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

---

## [실습] **Text2SQL RAG 구현**

- "ETFs" 테이블만 선택하여 RAG 시스템을 구현하고 테스트 (ignore_tables 옵션)
- LangSmith Trace를 분석하여 작동 방식 이해 

In [ ]:
# 여기에 코드를 작성하세요.